<a href="https://colab.research.google.com/github/Ritiha-VS/Smart-Crop-Prediction-and-Disease-Detection/blob/main/Smart_crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("CROP PREDICTION MODEL")

crop_df = pd.read_csv('/content/drive/MyDrive/Crop_recommendation.csv')

print("\nDataset Shape:", crop_df.shape)
print("\nSample Data:")
print(crop_df.head())
print("\nCrops:", crop_df['label'].unique())

plt.figure(figsize=(14, 5))
crop_df['label'].value_counts().plot(kind='bar', color='steelblue')
plt.title('Crop Distribution in Dataset')
plt.xlabel('Crop')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/smart_crop_project/crop_distribution.png')
plt.show()

X_crop = crop_df.drop('label', axis=1)
y_crop = crop_df['label']
le_crop = LabelEncoder()
y_crop_encoded = le_crop.fit_transform(y_crop)
scaler_crop = StandardScaler()
X_crop_scaled = scaler_crop.fit_transform(X_crop)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_crop_scaled, y_crop_encoded, test_size=0.2, random_state=42
)

crop_model = RandomForestClassifier(n_estimators=100, random_state=42)
crop_model.fit(X_train_c, y_train_c)
y_pred_c = crop_model.predict(X_test_c)
acc_crop = accuracy_score(y_test_c, y_pred_c)

print(f"\nCrop Prediction Accuracy: {acc_crop * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_c, target_names=le_crop.classes_))

cm_crop = confusion_matrix(y_test_c, y_pred_c)
sns.heatmap(cm_crop, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_crop.classes_, yticklabels=le_crop.classes_)
plt.title('Crop Prediction - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/smart_crop_project/crop_confusion_matrix.png')
plt.show()

print("\n── Sample Crop Prediction ──")

sample_crop = np.array([[90, 42, 43, 20.8, 82.0, 6.5, 202.9]])
sample_crop_scaled = scaler_crop.transform(sample_crop)
predicted_crop = le_crop.inverse_transform(crop_model.predict(sample_crop_scaled))

print(f"Input  → N=90, P=42, K=43, Temp=20.8°C, Humidity=82%, pH=6.5, Rainfall=202.9mm")
print(f"Predicted Crop → {predicted_crop[0].upper()}")

print("\nPLANT DISEASE DETECTION FROM SENSOR READINGS")
def label_disease(row):
    if row['temperature'] > 35 and row['humidity'] > 80:
        return 'Fungal Disease'
    elif row['ph'] < 5.5:
        return 'Nutrient Deficiency'
    elif row['N'] < 20 and row['P'] < 20 and row['K'] < 20:
        return 'Nutrient Deficiency'
    elif row['rainfall'] > 250 and row['ph'] < 6.0:
        return 'Root Rot'
    elif row['temperature'] < 10:
        return 'Cold Stress'
    else:
        return 'Healthy'

disease_df = crop_df.copy()
disease_df['disease'] = disease_df.apply(label_disease, axis=1)

print("\nDisease Label Distribution:")
print(disease_df['disease'].value_counts())

disease_df['disease'].value_counts().plot(kind='bar', color='coral')
plt.title('Disease Distribution from Sensor Readings')
plt.xlabel('Condition')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/smart_crop_project/disease_distribution.png')
plt.show()

X_disease = disease_df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y_disease = disease_df['disease']

le_disease = LabelEncoder()
y_disease_encoded = le_disease.fit_transform(y_disease)

scaler_disease = StandardScaler()
X_disease_scaled = scaler_disease.fit_transform(X_disease)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_disease_scaled, y_disease_encoded, test_size=0.2, random_state=42
)

disease_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
disease_model.fit(X_train_d, y_train_d)

y_pred_d = disease_model.predict(X_test_d)
acc_disease = accuracy_score(y_test_d, y_pred_d)

print(f"\nDisease Detection Accuracy: {acc_disease * 100:.2f}%")
print("\nClassification Report:")

present_labels = np.unique(np.concatenate([y_test_d, y_pred_d]))
present_names  = le_disease.inverse_transform(present_labels)

print(classification_report(y_test_d, y_pred_d,
                            labels=present_labels,
                            target_names=present_names))

cm_disease = confusion_matrix(y_test_d, y_pred_d, labels=present_labels)

sns.heatmap(cm_disease, annot=True, fmt='d', cmap='Reds',
            xticklabels=present_names, yticklabels=present_names)
plt.title('Disease Detection - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/smart_crop_project/disease_confusion_matrix.png')
plt.show()

print("\n── Sample Disease Predictions ──")

test_cases = [
    [30, 15, 20, 38.0, 90.0, 6.0, 150.0],
    [10, 8,  5,  25.0, 60.0, 4.8, 100.0],
    [80, 40, 40, 22.0, 70.0, 6.8, 180.0],
    [50, 30, 35, 28.0, 75.0, 5.5, 280.0],
]

labels_input = [
    "High Temp+Humidity",
    "Low pH + Low NPK",
    "Balanced Conditions",
    "High Rainfall + Low pH"
]

print(f"\n{'Condition':<25} {'Predicted Disease':<25}")
for tc, label in zip(test_cases, labels_input):
    inp = np.array([tc])
    inp_scaled = scaler_disease.transform(inp)
    pred = le_disease.inverse_transform(disease_model.predict(inp_scaled))
    print(f"{label:<25} {pred[0]:<25}")

print("\nFINAL SUMMARY")
print(f"Crop Prediction Accuracy    : {acc_crop * 100:.2f}%")
print(f"Disease Detection Accuracy  : {acc_disease * 100:.2f}%")